# 01 — Data Exploration
EDA on Financial PhraseBank (FPB) and FiQA-SA: class distributions, text lengths, sample inspection.

In [ ]:
import sys, os
# Colab: uncomment and set PROJECT_DIR
# PROJECT_DIR = '/content/drive/MyDrive/finllama-sentiment'
# sys.path.insert(0, PROJECT_DIR); os.chdir(PROJECT_DIR)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_fpb, load_fiqa
from src.utils import load_config, set_seed

cfg = load_config()
set_seed(cfg['seed'])
sns.set_theme(style='whitegrid')

## Load datasets

In [ ]:
fpb_train, fpb_test = load_fpb(
    config=cfg['datasets']['fpb']['config'],
    test_fraction=cfg['datasets']['fpb']['test_fraction'],
    seed=cfg['seed'],
)
fiqa = load_fiqa(neutral_band=cfg['datasets']['fiqa']['neutral_band'])

df_fpb  = pd.DataFrame(fpb_train + fpb_test)
df_fiqa = pd.DataFrame(fiqa)
print(f'FPB  train={len(fpb_train):,}  test={len(fpb_test):,}')
print(f'FiQA test={len(fiqa):,}')

## Class distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
label_order = ['negative', 'neutral', 'positive']
palette = {'negative': '#d62728', 'neutral': '#aec7e8', 'positive': '#2ca02c'}

for ax, (df, title) in zip(axes, [(df_fpb, 'FPB'), (df_fiqa, 'FiQA-SA')]):
    counts = df['label'].value_counts().reindex(label_order)
    counts.plot.bar(ax=ax, color=[palette[l] for l in label_order], edgecolor='black')
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=0)
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Class Distributions', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../presentation/key_figures/class_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Text-length distributions

In [ ]:
for df, name in [(df_fpb, 'FPB'), (df_fiqa, 'FiQA')]:
    lengths = df['text'].str.split().str.len()
    print(f'{name}  median={lengths.median():.0f}  p95={lengths.quantile(0.95):.0f}  max={lengths.max()}')

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, (df, name) in zip(axes, [(df_fpb, 'FPB'), (df_fiqa, 'FiQA-SA')]):
    df['text'].str.split().str.len().hist(bins=40, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'{name} — word count')
    ax.set_xlabel('Words')
plt.tight_layout()
plt.show()

## Sample inspection — one per class per dataset

In [ ]:
for name, df in [('FPB', df_fpb), ('FiQA', df_fiqa)]:
    print(f'\n=== {name} ===')
    for lbl in ['negative', 'neutral', 'positive']:
        row = df[df['label'] == lbl].sample(1, random_state=42).iloc[0]
        print(f'  [{lbl}] {row["text"][:120]}')